# Step 2: Autograd

## 2.1 Ne Değişiyor?

Bu bölümde aynı Türkçe isim veri kümesini, tokenizer'ı ve MLP
mimarisini kullanacağım. Modelde 3424 öğrenilebilir parametre
bulunacak ve eğitim yine SGD ile yapılacak.

Değişen kısım gradyanların hesaplanmasıdır. Önceki bölümde geri
yayılımı her katman için elle yazmıştım. Bu bölümde Value sınıfı
ile sayılar üzerinde yapılan işlemleri kaydedeceğim.

loss.backward() bu işlemleri geriye doğru takip ederek zincir
kuralını uygulayacak ve gradyanları otomatik hesaplayacak.

In [1]:
import random
import math

docs = [
    l.strip().replace(" ", "").replace("İ", "i").replace("I", "ı").lower()
    for l in open("../data/raw_names.txt", encoding="utf-8")
        .read().strip().split("\n")
    if l.strip()
]

random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

In [2]:
print("İsim sayısı:", len(docs))
print("Karakter sayısı:", len(uchars))
print("BOS ID:", BOS)
print("Vocabulary boyutu:", vocab_size)

İsim sayısı: 1100
Karakter sayısı: 29
BOS ID: 29
Vocabulary boyutu: 30


## 2.2 Value Sınıfı

Autograd için normal sayılar yerine Value nesneleri kullanacağız.
Her nesne sayısal değerini, gradyanını, hangi değerlerden üretildiğini
ve bu değerlere göre yerel türevlerini saklayacak.

Bu bilgiler, hesaplama grafiğini oluşturmak ve daha sonra zincir
kuralıyla gradyanları otomatik hesaplamak için kullanılacak.

In [3]:
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data)
        )

IndentationError: expected an indented block after function definition on line 9 (712259145.py, line 10)

In [ ]:
a = Value(3.0)

print("Değer:", a.data)
print("Gradyan:", a.grad)
print("Girdiler:", a._children)
print("Yerel türevler:", a._local_grads)


b = Value(2.0)

toplam = a + b
carpim = a * b

print("Toplam:", toplam.data)
print("Toplamın girdileri:", [v.data for v in toplam._children])
print("Toplamın yerel türevleri:", toplam._local_grads)

print()

print("Çarpım:", carpim.data)
print("Çarpımın girdileri:", [v.data for v in carpim._children])
print("Çarpımın yerel türevleri:", carpim._local_grads)
print("Çarpımın gradyanı:", carpim.grad)